<div class="alert">Show ssHG1G2 convergence vs parameters</div>

In [1]:
import os
import time
import sys

sys.path.append("..")


import numpy as np

import ssptools
from astropy.coordinates import SkyCoord
import rocks

from fink_utils.sso.spins import (
    estimate_sso_params,
    func_sshg1g2,
)  # , func_hg1g2, cos_aspect_angle
from fink_utils.sso.periods import estimate_synodic_period
from fink_utils.sso.utils import compute_light_travel_correction, estimate_axes_ratio

import matplotlib.pyplot as plt

import pandas as pd

# import seaborn as sns
# sns.set_context("poster")

In [2]:
fink_colors = ["#15284F", "#F5622E"]

# Functions to add to fink-utils

In [3]:
def angle_between_vectors(v1, v2):
    """
    Compute the angle between two 3D vectors.

    Parameters
    ----------
    v1 : list or np.ndarray
        The first 3D vector.
    v2 : list or np.ndarray
        The second 3D vector.

    Returns
    -------
    float
        The angle between the two vectors in radians.
    """
    v1 = np.array(v1)
    v2 = np.array(v2)

    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    cos_theta = dot_product / (norm_v1 * norm_v2)
    angle = np.arccos(np.clip(cos_theta, -1.0, 1.0))  # Clip to handle numerical issues

    return angle


# Example usage
# v1 = [1.0, 0, 0]
# v2 = [-1.0, -1.0, 1]
# angle = angle_between_vectors(v1, v2)
# print(f"The angle between the vectors is {np.degrees(angle):.2f} degrees")


def synodic_to_sidereal(synodic_period, X):
    """
    Convert synodic rotation period to sidereal rotation period.

    TBD

    Parameters
    ----------
    synodic_period : float
        Synodic rotation period in days.

    Returns
    -------
    sidereal_period : float
        Sidereal rotation period in days.
    """
    sidereal_period = synodic_period
    return sidereal_period

# Target definition

In [4]:
ssnamenr = 5209
ssnamenr = 136108  # KBO = fixed = Haumea
# ssnamenr = 186153 # High frequency observation
# ssnamenr = 223 # Example article
# ssnamenr = 9799 # JTO lotta obs

# ssnamenr = 17365 # 1978VF11 Thymbareus

ssnamenr = 1139
ssnamenr = 22

ssocard = rocks.Rock(ssnamenr)
name, num = ssocard.name, ssocard.number
name, num

('Kalliope', 22)

## Period estimation

In [5]:
# pdf.to_parquet('sylvia.parquet')
# pdf = pd.read_parquet("sylvia.parquet")

# toto = pdf.copy()

In [6]:
flavor = "SHG1G2"

period_range = (1 / 24, 7)  # 1hour to 7 days

t0 = time.time()
period, chi2red, frequency, power, model, pdf = estimate_synodic_period(
    ssnamenr,
    # pdf=pdf,
    flavor=flavor,
    Nterms_base=1,
    period_range=period_range,
    return_extra_info=True,
)

In [7]:
print(
    "[{:.2f} seconds] model={}: period={:.2f} hours (chi2red={:.2f}) -- SsODNet: period={:.2f} h ({:s})".format(
        time.time() - t0,
        flavor,
        period,
        chi2red,
        ssocard.spin[0].period.value,
        ",".join(ssocard.spin[0].bibref.bibcode),
    )
)

[5.50 seconds] model=SHG1G2: period=6.06 hours (chi2red=14.49) -- SsODNet: period=4.15 h (2022A&A...662A..71F,2017A&A...601A.114H,2021A&A...654A..56V)


# Fit sHG1G2

In [8]:
# H G1 G2 estimate
fit_shg1g2 = estimate_sso_params(
    pdf["i:magpsf_red"],
    pdf["i:sigmapsf"],
    np.radians(pdf["Phase"]),
    pdf["i:fid"],
    ra=np.radians(pdf["i:ra"]),
    dec=np.radians(pdf["i:dec"]),
    model="SHG1G2",
)

# Store residuals
pdf["res_sHG1G2"] = pdf["residuals"]

# Fit ssHG1G2

In [9]:
jd = pdf["i:jd"].tolist()

eph = ssptools.ephemcc(name, jd, tcoor=2, observer="500")
# eph.to_csv("ephemeris.csv", index=False)
# eph = pd.read_csv("ephemeris.csv")

pdf["jd_ltc"] = compute_light_travel_correction(pdf["i:jd"], eph["Dobs"])

In [10]:
# Initial guess
ra0 = np.radians(ssocard.spin[0].RA0.value)
de0 = np.radians(ssocard.spin[0].DEC0.value)
Ps = ssocard.spin[0].period.value / 24.0
# Ps = 8.16827 / 24  # Lutetia

p0 = [
    fit_shg1g2["H_1"],
    fit_shg1g2["G1_1"],
    fit_shg1g2["G2_1"],
    ra0,
    de0,
    Ps,
    fit_shg1g2["a_b"],
    fit_shg1g2["a_c"],
    0.0,
]

# Constrained Fit
fit_sshg1g2 = estimate_sso_params(
    pdf["i:magpsf_red"],
    pdf["i:sigmapsf"],
    np.radians(pdf["Phase"]),
    pdf["i:fid"],
    ra=np.radians(pdf["i:ra"]),
    dec=np.radians(pdf["i:dec"]),
    jd=pdf["jd_ltc"],
    model="SSHG1G2",
    p0=p0,
)

print(fit_sshg1g2['status'])

3


In [18]:
Ps = ssocard.spin[0].period.value / 24.0
Ps*24

4.1482

In [ ]:
ra0 = np.radians(ssocard.spin[0].RA0.value)
de0 = np.radians(ssocard.spin[0].DEC0.value)
Ps = ssocard.spin[0].period.value / 24.0

ra0 = (np.radians(fit_sshg1g2["alpha0"]) + np.pi) % (2 * np.pi)
de0 = -np.radians(fit_sshg1g2["delta0"])
# Ps = ssocard.spin[0].period.value / 24.0

ab, ac = fit_sshg1g2["a_b"], fit_sshg1g2["a_c"]
phi0 = fit_sshg1g2["phi0"]

simu = {
    "H": {"min": -1, "max": 1, "N": 100},
    "G1": {"min": -1.2, "max": 1.2, "N": 100},
    "G2": {"min": -1.2, "max": 1.2, "N": 100},
    "Ps": {"min": -5e-4, "max": 5e-4, "N": 1000},
    "ra0": {"min": -20, "max": 20, "N": 100},
    "de0": {"min": -20, "max": 20, "N": 100},
    "ab": {"min": -2, "max": 2, "N": 100},
    "ac": {"min": -2, "max": 2, "N": 100},
    "phi0": {"min": 0, "max": 360, "N": 720},
}

results = pd.DataFrame(
    columns=[
        "set",
        "H",
        "G1",
        "G2",
        "ra0",
        "de0",
        "Ps",
        "ab",
        "ac",
        "phi0",
        "rms",
        "fid",
    ],
    index=None,
)


k = 0
for index, filtername in enumerate([1, 2]):

    # Single band
    mask = pdf["i:fid"] == filtername
    pha = np.vstack(
        [
            np.radians(pdf.loc[mask, "Phase"]),
            np.radians(pdf.loc[mask, "i:ra"]),
            np.radians(pdf.loc[mask, "i:dec"]),
            pdf.loc[mask, "jd_ltc"],
        ]
    )

    # Per-band parameter
    H, G1, G2 = (
        fit_sshg1g2[f"H_{filtername}"],
        fit_sshg1g2[f"G1_{filtername}"],
        fit_sshg1g2[f"G2_{filtername}"],
    )

    # H dependance
    H_arr = np.linspace(H + simu["H"]["min"], H + simu["H"]["max"], simu["H"]["N"])
    for trial_H in H_arr:
        model = func_sshg1g2(pha, trial_H, G1, G2, ra0, de0, Ps, ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = trial_H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = ra0
        results.loc[k, "de0"] = de0
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "H"
        k = k + 1

    # G1 dependance
    min_G1 = np.max([0, G1 + simu["G1"]["min"]])
    max_G1 = np.min([1, G1 + simu["G1"]["max"]])
    G_arr = np.linspace(min_G1, max_G1, simu["G1"]["N"])
    for i, trial_G in enumerate(G_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, trial_G, G2, ra0, de0, Ps, ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = trial_G
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = ra0
        results.loc[k, "de0"] = de0
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "G1"
        k = k + 1

    # G2 dependance
    min_G2 = np.max([0, G2 + simu["G2"]["min"]])
    max_G2 = np.min([1, G2 + simu["G2"]["max"]])
    G_arr = np.linspace(min_G2, max_G2, simu["G2"]["N"])
    for i, trial_G in enumerate(G_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, trial_G, ra0, de0, Ps, ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = trial_G
        results.loc[k, "ra0"] = ra0
        results.loc[k, "de0"] = de0
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "G2"
        k = k + 1

    # Period dependance
    P_arr = np.linspace(Ps + simu["Ps"]["min"], Ps + simu["Ps"]["max"], simu["Ps"]["N"])
    for i, trial_Ps in enumerate(P_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, G2, ra0, de0, trial_Ps, ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = ra0
        results.loc[k, "de0"] = de0
        results.loc[k, "Ps"] = trial_Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "Ps"
        k = k + 1

    # RA dependance
    RA_arr = np.radians(
        np.linspace(
            np.degrees(ra0) + simu["ra0"]["min"],
            np.degrees(ra0) + simu["ra0"]["max"],
            simu["ra0"]["N"],
        )
    )
    for i, trial_ra0 in enumerate(RA_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, G2, trial_ra0, de0, Ps, ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        # results.loc[k, "ra0"] = np.degrees(trial_ra0 % (2 * np.pi))
        results.loc[k, "ra0"] = np.degrees(trial_ra0)
        results.loc[k, "de0"] = np.degrees(de0)
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "ra0"
        k = k + 1

    # DEC dependance
    min_DE = np.max([-90, np.degrees(de0) + simu["de0"]["min"]])
    max_DE = np.min([90, np.degrees(de0) + simu["de0"]["max"]])
    DE_arr = np.radians(np.linspace(min_DE, max_DE, simu["de0"]["N"]))
    for i, trial_de0 in enumerate(DE_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, G2, ra0, trial_de0, Ps, ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = np.degrees(ra0)
        results.loc[k, "de0"] = np.degrees(trial_de0)
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "de0"
        k = k + 1

    # ab dependance
    min_ab = np.max([1, ab + simu["ab"]["min"]])
    max_ab = np.min([5, ab + simu["ab"]["max"]])
    ab_arr = np.linspace(min_ab, max_ab, simu["ab"]["N"])
    for i, trial_ab in enumerate(ab_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, G2, ra0, de0, Ps, trial_ab, ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = np.degrees(ra0)
        results.loc[k, "de0"] = np.degrees(de0)
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = trial_ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "ab"
        k = k + 1

    # ac dependance
    min_ac = np.max([1, ac + simu["ac"]["min"]])
    max_ac = np.min([5, ac + simu["ac"]["max"]])
    ac_arr = np.linspace(min_ac, max_ac, simu["ac"]["N"])
    for i, trial_ac in enumerate(ac_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, G2, ra0, de0, Ps, ab, trial_ac, phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = np.degrees(ra0)
        results.loc[k, "de0"] = np.degrees(de0)
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = trial_ac
        results.loc[k, "phi0"] = phi0
        results.loc[k, "set"] = "ac"
        k = k + 1

    # phi dependance
    phi_arr = np.radians(np.linspace(simu["phi0"]["min"], simu["phi0"]["max"], simu["phi0"]["N"]))
    for i, trial_phi0 in enumerate(phi_arr):
        # Compute the model
        model = func_sshg1g2(pha, H, G1, G2, ra0, de0, Ps, ab, ac, trial_phi0)
        rms = np.sqrt(np.mean((model - pdf.loc[mask, "i:magpsf_red"]) ** 2))
        results.loc[k, "rms"] = rms
        results.loc[k, "fid"] = filtername

        results.loc[k, "H"] = H
        results.loc[k, "G1"] = G1
        results.loc[k, "G2"] = G2
        results.loc[k, "ra0"] = np.degrees(ra0)
        results.loc[k, "de0"] = np.degrees(de0)
        results.loc[k, "Ps"] = Ps
        results.loc[k, "ab"] = ab
        results.loc[k, "ac"] = ac
        results.loc[k, "phi0"] = np.degrees(trial_phi0)
        results.loc[k, "set"] = "phi0"
        k = k + 1

    # initial conditions for that filter
    results.loc[k, "H"] = H
    results.loc[k, "G1"] = G1
    results.loc[k, "G2"] = G2
    results.loc[k, "ra0"] = np.degrees(ra0)
    results.loc[k, "de0"] = np.degrees(de0)
    results.loc[k, "Ps"] = Ps
    results.loc[k, "ab"] = ab
    results.loc[k, "ac"] = ac
    results.loc[k, "phi0"] = phi0
    results.loc[k, "set"] = f"best_{filtername}"
    k = k + 1

In [22]:
sys.path.append("..")
import figure_setup as fs

In [23]:
fig, ax = plt.subplots(
    2,
    3,
    figsize=fs.figsize(1),
    gridspec_kw={
        "wspace": 0.18,
        "left": 0.07,
        "right": 0.99,
        "top": 0.99,
        "bottom": 0.07,
    },
)


labels = {
    "H": "H",
    "G1": "$G_1$",
    "G2": "$G_2$",
    "ra0": r"$\alpha_0$",
    "de0": r"$\delta_0$",
    "Ps": "Period",
    "ab": "a/b",
    "ac": "a/c",
    "phi0": r"$\phi_0$",
}
fname = ["g", "r"]
for fid in [1, 2]:

    col = f"C{fid-1}"
    col = fink_colors[fid - 1]

    # H G1G2 ab/ac
    for i, p in enumerate(["H", "G1", "ab"]):
        cond = (results["set"] == p) & (results["fid"] == fid)

        lbl = ""
        if p == "H":
            lbl = fname[fid - 1]
        if (fid == 1) & (p == "G1"):
            lbl = labels[p]
        if (fid == 1) & (p == "ab"):
            lbl = labels[p]
        ax[0, i].plot(
            results.loc[cond, p],
            results.loc[cond, "rms"],
            color=col,
            # label=labels[p]
            label=lbl,
        )
    for i, p in enumerate(["G2", "ac"]):
        cond = (results["set"] == p) & (results["fid"] == fid)

        if (fid == 1) & (p == "G2"):
            lbl = labels[p]
        if (fid == 1) & (p == "ac"):
            lbl = labels[p]
        ax[0, i + 1].plot(
            results.loc[cond, p],
            results.loc[cond, "rms"],
            color=col,
            label=lbl,
            linestyle="--",
        )

    # RA
    p = "ra0"
    cond = (results["set"] == p) & (results["fid"] == fid)
    ax[1, 0].plot(
        results.loc[cond, p], results.loc[cond, "rms"], color=col, label=labels[p]
    )

    # DEC
    p = "de0"
    cond = (results["set"] == p) & (results["fid"] == fid)
    ax[1, 1].plot(
        results.loc[cond, p], results.loc[cond, "rms"], color=col, label=labels[p]
    )

    # Period
    p = "Ps"
    cond = (results["set"] == p) & (results["fid"] == fid)

    ref_per = 0.215885
    ref_per = results.loc[cond, p].min()
    plot_per = (results.loc[cond, p] - ref_per) * 86400

    ax[1, 2].plot(plot_per, results.loc[cond, "rms"], color=col, label=labels[p])


# Axes
for a in ax[:, 0]:
    a.set_ylabel("RMS")

ax[0, 0].set_xlabel("H")
ax[0, 1].set_xlabel("$G_1$, $G_2$")
ax[0, 2].set_xlabel("a/b, a/c")

ax[1, 0].set_xlabel(r"Right ascension ($^o$)")
ax[1, 1].set_xlabel(r"Declination ($^o$)")
ax[1, 2].set_xlabel(f"Sidereal period (s from {ref_per*24:.4f}h)")

# Sylvia specific
ax[0,1].set_ylim(top=1) # GG
# ax[0,2].set_ylim(top=0.3) # ab ac
# ax[1,0].set_ylim(top=0.07) # ra0


# Legend
ax[0, 0].legend(loc="upper center")
ax[0, 1].legend(loc="upper right")
ax[0, 2].legend(loc="upper left")

# ax[1,0].set_xlim(320,360)

fig.savefig("socca_convergence.pgf")
fig.savefig("socca_convergence.png")

In [18]:
np.min(P_arr), np.max(P_arr), np.max(P_arr)-np.min(P_arr), (np.max(P_arr)-np.min(P_arr))*24, (np.max(P_arr)-np.min(P_arr))*24*60

(0.17234166666666667,
 0.17334166666666667,
 0.0010000000000000009,
 0.02400000000000002,
 1.4400000000000013)

In [161]:
fig, ax = plt.subplots(gridspec_kw={"bottom":0.15})


for fid in [1, 2]:

    col = f"C{fid-1}"
    col = fink_colors[fid - 1]

    p = "phi0"
    cond = (results["set"] == p) & (results["fid"] == fid)
    ax.plot(results.loc[cond, p], results.loc[cond, "rms"], color=col, label=labels[p])

ax.set_ylabel("RMS")
ax.set_xlabel(r"$\phi_0$")


# fig.savefig("ssHG1G2_convergence.pgf")
fig.savefig("test_phi0.png")

In [145]:
    cond = (results["set"] == p) & (results["fid"] == fid)


In [148]:
results.loc[cond,'phi0']

1700           0.0
1701      0.500695
1702      1.001391
1703      1.502086
1704      2.002782
           ...    
2415    357.997218
2416    358.497914
2417    358.998609
2418    359.499305
2419         360.0
Name: phi0, Length: 720, dtype: object

In [81]:
np.linspace(
            np.degrees(ra0) + simu["ra0"]["min"],
            np.degrees(ra0) + simu["ra0"]["max"],
            8,
        )

array([328.20543126, 333.91971697, 339.63400269, 345.3482884 ,
       351.06257412, 356.77685983, 362.49114555, 368.20543126])